<a href="https://colab.research.google.com/github/LouiseHjorth/speciale-forecasting-hay/blob/main/Forecasting_speciale_ML_modeller.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Indlæsning af data

In [ ]:
from google.colab import files
uploaded = files.upload()

import pandas as pd
import numpy as np

from sklearn.model_selection import TimeSeriesSplit, GridSearchCV
from sklearn.linear_model import ElasticNet
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor

from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

excel_file = pd.ExcelFile('Speciale Data.xlsx')
print("Ark i Speciale Data.xlsx:", excel_file.sheet_names)

df_internal = pd.read_excel('Speciale Data.xlsx', sheet_name='Speciale Data')
df_external = pd.read_excel('Eksterne variable.xlsx')

# Data rensning

In [ ]:
df_internal.columns = df_internal.columns.str.strip()
df_external.columns = df_external.columns.str.strip()

df_external = df_external.rename(columns={'Time': 'Date'})

df_internal['Date'] = pd.to_datetime(df_internal['Date'])
df_external['Date'] = pd.to_datetime(df_external['Date'], format='%d.%m.%Y')

df_internal = df_internal[
    [
        'Sold Quantity',
        'Size Name',
        'Date',
        'UniqueCountryCount',
        'customers_with_purchase',
        'customers_multi_buyer',
        'avg_order_count_per_customer',
        'ActiveColors'
    ]
].copy()

df_external = df_external[
    [
        'Date',
        'Colour crate',
        'Møbel indeks'
    ]
].copy()

df_internal = df_internal.dropna(subset=['Sold Quantity', 'Size Name', 'Date']).copy()

df = df_internal.merge(df_external, on='Date', how='left')
df = df.sort_values(['Size Name', 'Date']).reset_index(drop=True)

print("\nManglende værdier efter merge:")
print(df.isna().sum())

print("\nAntal observationer pr. størrelse:")
print(df['Size Name'].value_counts())

# Opretter dummy variable og sæson varaible

## Three days of design

In [ ]:
df['tdod_t'] = (df['Date'].dt.month == 6).astype(int)
df['tdod_t1'] = (df['Date'].dt.month == 7).astype(int)


## Launch varaible

In [ ]:
launch_dates = pd.to_datetime([
    '2023-01-01',
    '2023-03-01',
    '2025-06-01'
])

df['launch_t'] = df['Date'].isin(launch_dates).astype(int)
df['launch_t1'] = df.groupby('Size Name')['launch_t'].shift(1).fillna(0).astype(int)

## Sæson varaible

In [ ]:
df['month'] = df['Date'].dt.month
df['month_sin'] = np.sin(2 * np.pi * df['month'] / 12)
df['month_cos'] = np.cos(2 * np.pi * df['month'] / 12)


# Lags af variable

In [ ]:
df['customers_with_purchase_lag1'] = df.groupby('Size Name')['customers_with_purchase'].shift(1)
df['customers_multi_buyer_lag1'] = df.groupby('Size Name')['customers_multi_buyer'].shift(1)
df['avg_order_count_per_customer_lag1'] = df.groupby('Size Name')['avg_order_count_per_customer'].shift(1)
df['UniqueCountryCount_lag1'] = df.groupby('Size Name')['UniqueCountryCount'].shift(1)

df['Colour_crate_lag1'] = df.groupby('Size Name')['Colour crate'].shift(1)
df['Mobel_indeks_lag1'] = df.groupby('Size Name')['Møbel indeks'].shift(1)

# Splitter variable i størrelser

In [ ]:
df_small = df[df['Size Name'] == 'SMALL'].copy()
df_medium = df[df['Size Name'] == 'MEDIUM'].copy()
df_large = df[df['Size Name'] == 'LARGE'].copy()

print("\nShapes:")
print("Small :", df_small.shape)
print("Medium:", df_medium.shape)
print("Large :", df_large.shape)

# Forecast horisonter

In [ ]:
def create_targets(data):
    data = data.sort_values('Date').copy()
    data['target_3'] = data['Sold Quantity'].shift(-3)
    data['target_6'] = data['Sold Quantity'].shift(-6)
    data['target_12'] = data['Sold Quantity'].shift(-12)
    return data

df_small = create_targets(df_small)
df_medium = create_targets(df_medium)
df_large = create_targets(df_large)


# Forklarende varaible

In [ ]:
selected_features = [
    'UniqueCountryCount_lag1',
    'customers_with_purchase_lag1',
    'customers_multi_buyer_lag1',
    'avg_order_count_per_customer_lag1',
    'ActiveColors',
    'Colour_crate_lag1',
    'Mobel_indeks_lag1',
    'tdod_t',
    'tdod_t1',
    'launch_t',
    'launch_t1',
    'month_sin',
    'month_cos'
]


# Klargøring af model datasæt

In [ ]:
def prepare_ml_dataset(data, target_col):
    cols_needed = ['Date', 'Sold Quantity'] + selected_features + [target_col]
    dataset = data[cols_needed].copy()
    dataset = dataset.dropna().reset_index(drop=True)
    return dataset

small_3 = prepare_ml_dataset(df_small, 'target_3')
small_6 = prepare_ml_dataset(df_small, 'target_6')
small_12 = prepare_ml_dataset(df_small, 'target_12')

medium_3 = prepare_ml_dataset(df_medium, 'target_3')
medium_6 = prepare_ml_dataset(df_medium, 'target_6')
medium_12 = prepare_ml_dataset(df_medium, 'target_12')

large_3 = prepare_ml_dataset(df_large, 'target_3')
large_6 = prepare_ml_dataset(df_large, 'target_6')
large_12 = prepare_ml_dataset(df_large, 'target_12')

print("\nShapes på ML-datasæt:")
print("small_3 :", small_3.shape)
print("small_6 :", small_6.shape)
print("small_12:", small_12.shape)
print("medium_3:", medium_3.shape)
print("medium_6:", medium_6.shape)
print("medium_12:", medium_12.shape)
print("large_3 :", large_3.shape)
print("large_6 :", large_6.shape)
print("large_12:", large_12.shape)

## Samler datasæt

In [ ]:
datasets = {
    'small_3': (small_3, 'target_3'),
    'small_6': (small_6, 'target_6'),
    'small_12': (small_12, 'target_12'),
    'medium_3': (medium_3, 'target_3'),
    'medium_6': (medium_6, 'target_6'),
    'medium_12': (medium_12, 'target_12'),
    'large_3': (large_3, 'target_3'),
    'large_6': (large_6, 'target_6'),
    'large_12': (large_12, 'target_12')
}


# Vælger antal split til cross validation

In [ ]:
def get_tscv(n_obs):
    if n_obs >= 36:
        return TimeSeriesSplit(n_splits=3)
    elif n_obs >= 24:
        return TimeSeriesSplit(n_splits=2)
    else:
        return None


# Model funktioner

In [ ]:
def run_elastic_net_cv(X, y, tscv):
    pipe = Pipeline([
        ('scaler', StandardScaler()),
        ('model', ElasticNet(random_state=42, max_iter=10000))
    ])

    param_grid = {
        'model__alpha': [0.01, 0.1, 1.0, 10.0],
        'model__l1_ratio': [0.2, 0.5, 0.8, 1.0]
    }

    grid = GridSearchCV(
        estimator=pipe,
        param_grid=param_grid,
        scoring='neg_mean_absolute_error',
        cv=tscv,
        n_jobs=-1
    )

    grid.fit(X, y)
    return grid


def run_random_forest_cv(X, y, tscv):
    model = RandomForestRegressor(random_state=42)

    param_grid = {
        'n_estimators': [100, 200],
        'max_depth': [3, 5, None],
        'min_samples_split': [2, 5],
        'min_samples_leaf': [1, 2],
        'max_features': ['sqrt', 'log2']
    }

    grid = GridSearchCV(
        estimator=model,
        param_grid=param_grid,
        scoring='neg_mean_absolute_error',
        cv=tscv,
        n_jobs=-1
    )

    grid.fit(X, y)
    return grid


def run_xgb_cv(X, y, tscv):
    model = XGBRegressor(
        objective='reg:squarederror',
        random_state=42
    )

    param_grid = {
        'n_estimators': [100, 200],
        'max_depth': [2, 3, 5],
        'learning_rate': [0.01, 0.05, 0.1]
    }

    grid = GridSearchCV(
        estimator=model,
        param_grid=param_grid,
        scoring='neg_mean_absolute_error',
        cv=tscv,
        n_jobs=-1
    )

    grid.fit(X, y)
    return grid


# Kører modeller

In [ ]:
results_all = []
best_params_all = {}

for name, (data, target_col) in datasets.items():

    print(f"\nKører dataset: {name}")

    X = data.drop(columns=['Date', 'Sold Quantity', target_col])
    y = data[target_col]

    n_obs = len(data)
    tscv = get_tscv(n_obs)

    if tscv is None:
        print(f"Springer over {name}: for få observationer")
        continue

    print(f"Observationer: {n_obs}")

    en_grid = run_elastic_net_cv(X, y, tscv)
    en_cv_mae = -en_grid.best_score_

    results_all.append({
        'dataset': name,
        'model': 'Elastic Net',
        'n_obs': n_obs,
        'cv_mae': en_cv_mae
    })

    best_params_all[(name, 'Elastic Net')] = en_grid.best_params_

    rf_grid = run_random_forest_cv(X, y, tscv)
    rf_cv_mae = -rf_grid.best_score_

    results_all.append({
        'dataset': name,
        'model': 'Random Forest',
        'n_obs': n_obs,
        'cv_mae': rf_cv_mae
    })

    best_params_all[(name, 'Random Forest')] = rf_grid.best_params_

    xgb_grid = run_xgb_cv(X, y, tscv)
    xgb_cv_mae = -xgb_grid.best_score_

    results_all.append({
        'dataset': name,
        'model': 'XGBoost',
        'n_obs': n_obs,
        'cv_mae': xgb_cv_mae
    })

    best_params_all[(name, 'XGBoost')] = xgb_grid.best_params_

# Resultater

In [ ]:
results_df = pd.DataFrame(results_all)
results_df['cv_mae'] = results_df['cv_mae'].round(2)

print("\nAlle CV-resultater:")
print(results_df.sort_values(['dataset', 'model']))

## Bedste model pr. datasæt

In [ ]:
best_results = results_df.loc[
    results_df.groupby('dataset')['cv_mae'].idxmin()
].reset_index(drop=True)

print("\nBedste model pr. dataset:")
print(best_results.sort_values('dataset'))

## Pivot tabel

In [ ]:
pivot_results = results_df.pivot_table(
    index='dataset',
    columns='model',
    values='cv_mae'
).reset_index()

print("\nPivot-tabel:")
print(pivot_results)

final_results_table = pivot_results.copy()
final_results_table[['size', 'horizon']] = final_results_table['dataset'].str.split('_', expand=True)
final_results_table['horizon'] = final_results_table['horizon'].astype(int)

final_results_table = final_results_table[
    ['size', 'horizon', 'dataset', 'Elastic Net', 'Random Forest', 'XGBoost']
].sort_values(['size', 'horizon']).reset_index(drop=True)

final_results_table['best_model'] = final_results_table[['Elastic Net', 'Random Forest', 'XGBoost']].idxmin(axis=1)
final_results_table['best_cv_mae'] = final_results_table[['Elastic Net', 'Random Forest', 'XGBoost']].min(axis=1)

print("\nFinal CV-tabel:")
print(final_results_table)


## Printer bedst parametre

In [ ]:
print("\nBedste parametre:")
for key, value in best_params_all.items():
    print(f"{key}: {value}")

# Gemmer resultater

In [ ]:
results_df.to_excel("trin1_cv_results_all_models.xlsx", index=False)
best_results.to_excel("trin1_cv_best_model_per_dataset.xlsx", index=False)
pivot_results.to_excel("trin1_cv_pivot_results.xlsx", index=False)
final_results_table.to_excel("trin1_cv_final_results_table.xlsx", index=False)

print("\nFiler gemt:")
print("- trin1_cv_results_all_models.xlsx")
print("- trin1_cv_best_model_per_dataset.xlsx")
print("- trin1_cv_pivot_results.xlsx")
print("- trin1_cv_final_results_table.xlsx")

# ML forecast

In [ ]:
from sklearn.metrics import mean_absolute_error
from sklearn.linear_model import ElasticNet
from sklearn.ensemble import RandomForestRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from xgboost import XGBRegressor
import pandas as pd
import matplotlib.pyplot as plt


## Hjælpe funktioner

In [ ]:
def get_horizon_from_target(target_col):
    return int(target_col.split('_')[1])


def get_best_model(best_model_name, best_params):
    if best_model_name == "Elastic Net":
        model = Pipeline([
            ('scaler', StandardScaler()),
            ('model', ElasticNet(
                alpha=best_params['model__alpha'],
                l1_ratio=best_params['model__l1_ratio'],
                random_state=42,
                max_iter=10000
            ))
        ])

    elif best_model_name == "Random Forest":
        model = RandomForestRegressor(
            n_estimators=best_params['n_estimators'],
            max_depth=best_params['max_depth'],
            min_samples_split=best_params['min_samples_split'],
            min_samples_leaf=best_params['min_samples_leaf'],
            max_features=best_params['max_features'],
            random_state=42
        )

    elif best_model_name == "XGBoost":
        model = XGBRegressor(
            n_estimators=best_params['n_estimators'],
            max_depth=best_params['max_depth'],
            learning_rate=best_params['learning_rate'],
            objective='reg:squarederror',
            random_state=42
        )

    else:
        raise ValueError(f"Ukendt modelnavn: {best_model_name}")

    return model


def get_ml_split_date(forecast_start, horizon):
    forecast_start = pd.Timestamp(forecast_start)
    return forecast_start - pd.DateOffset(months=horizon)


def get_forecast_period_from_horizon(horizon):
    if horizon == 3:
        forecast_start = '2025-12-01'
        forecast_end = '2026-02-01'
    elif horizon == 6:
        forecast_start = '2025-09-01'
        forecast_end = '2026-02-01'
    elif horizon == 12:
        forecast_start = '2025-02-01'
        forecast_end = '2026-01-01'
    else:
        raise ValueError(f"Ukendt horisont: {horizon}")

    return forecast_start, forecast_end

## Forecast funktion

In [ ]:
def run_final_forecast(dataset_name, data, target_col, best_model_name, best_params):

    horizon = get_horizon_from_target(target_col)
    forecast_start, forecast_end = get_forecast_period_from_horizon(horizon)
    split_date = get_ml_split_date(forecast_start, horizon)

    print(f"\nKører FINAL forecast for: {dataset_name}")
    print(f"Horisont: {horizon} måneder")
    print(f"Forecast start: {forecast_start}")
    print(f"Forecast end: {forecast_end}")
    print(f"ML split_date: {split_date.date()}")

    train = data[data['Date'] < split_date].copy()
    test = data[data['Date'] >= split_date].copy()

    test['forecast_date'] = test['Date'] + pd.DateOffset(months=horizon)

    test = test[
        (test['forecast_date'] >= pd.Timestamp(forecast_start)) &
        (test['forecast_date'] <= pd.Timestamp(forecast_end))
    ].copy()

    if len(test) == 0:
        raise ValueError(
            f"Ingen testobservationer for {dataset_name}. "
            f"Tjek splitdato og forecastperiode."
        )

    X_train = train.drop(columns=['Date', 'Sold Quantity', target_col])
    y_train = train[target_col]

    X_test = test.drop(columns=['Date', 'Sold Quantity', target_col, 'forecast_date'])
    y_test = test[target_col]

    model = get_best_model(best_model_name, best_params)
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    mae = mean_absolute_error(y_test, y_pred)
    print(f"MAE: {mae:.2f}")
    print(f"Antal testobservationer: {len(test)}")

    forecast_df = test[['Date', 'forecast_date']].copy()
    forecast_df['actual'] = y_test.values
    forecast_df['forecast'] = y_pred
    forecast_df['error'] = forecast_df['actual'] - forecast_df['forecast']
    forecast_df['dataset'] = dataset_name
    forecast_df['model'] = best_model_name

    return forecast_df, mae

# Kører final forecast for alle datasæt

In [ ]:
final_results = []
final_forecasts = {}

for _, row in best_results.iterrows():
    dataset_name = row['dataset']
    model_name = row['model']

    data, target_col = datasets[dataset_name]
    best_params = best_params_all[(dataset_name, model_name)]

    forecast_df, mae = run_final_forecast(
        dataset_name=dataset_name,
        data=data,
        target_col=target_col,
        best_model_name=model_name,
        best_params=best_params
    )

    final_forecasts[dataset_name] = forecast_df

    final_results.append({
        'dataset': dataset_name,
        'model': model_name,
        'mae_ml': mae,
        'test_rows': len(forecast_df)
    })

final_results_df = pd.DataFrame(final_results)

print("\nFinal ML resultater:")
print(final_results_df.sort_values('dataset'))

In [ ]:
final_results_table = final_results_df.copy()
final_results_table[['size', 'horizon']] = final_results_table['dataset'].str.split('_', expand=True)
final_results_table['horizon'] = final_results_table['horizon'].astype(int)
final_results_table['mae_ml'] = final_results_table['mae_ml'].round(2)

final_results_table = final_results_table[
    ['size', 'horizon', 'dataset', 'model', 'test_rows', 'mae_ml']
].sort_values(['size', 'horizon']).reset_index(drop=True)

print("\nFinal ML tabel:")
print(final_results_table)

final_results_table.to_excel("trin2_final_ml_results.xlsx", index=False)

In [ ]:
from sklearn.metrics import mean_absolute_error
import pandas as pd

all_ml_forecasts = []

for dataset_name, (data, target_col) in datasets.items():

    print(f"Kører: {dataset_name}")

    for model_name in ["Elastic Net", "Random Forest", "XGBoost"]:

        best_params = best_params_all[(dataset_name, model_name)]

        forecast_df, mae = run_final_forecast(
            dataset_name=dataset_name,
            data=data,
            target_col=target_col,
            best_model_name=model_name,
            best_params=best_params
        )

        size, horizon = dataset_name.split("_")

        forecast_df["size"] = size
        forecast_df["horizon"] = int(horizon)
        forecast_df["Absolute Error"] = abs(
            forecast_df["actual"] - forecast_df["forecast"]
        )
        forecast_df["Error %"] = (
            forecast_df["Absolute Error"] / forecast_df["actual"] * 100
        )
        forecast_df["MAE"] = mae

        forecast_df = forecast_df[
            [
                "size",
                "horizon",
                "model",
                "forecast_date",
                "actual",
                "forecast",
                "error",
                "Absolute Error",
                "Error %",
                "MAE"
            ]
        ]

        all_ml_forecasts.append(forecast_df)

ml_forecast_summary = pd.concat(
    all_ml_forecasts,
    ignore_index=True
)

ml_forecast_summary = ml_forecast_summary.sort_values(
    ["size", "horizon", "model", "forecast_date"]
)

ml_forecast_summary = ml_forecast_summary.round(1)

print("\n Samlet ML forecasttabel ")
print(ml_forecast_summary.to_string(index=False))